In [49]:
import numpy as np
from scipy.optimize import linprog, minimize_scalar

In [50]:
def f(x):
    return x[0] - x[1]

In [51]:
def constraint(x):
    return x[0] - 2 * x[1] - 5

In [52]:
x0 = np.array([1, 1])

In [53]:
def approx_gradient(func, x, eps=1e-8):
    n = len(x)
    grad = np.zeros(n)
    for i in range(n):
        dx = np.zeros(n)
        dx[i] = eps
        grad[i] = (func(x + dx) - func(x - dx)) / (2 * eps)
    return grad

## Метод множителей Лагранжа

In [54]:
A = np.array([[3, 1]])
b = np.array([1])

In [55]:
import numpy as np
from scipy.optimize import fsolve

def lagrange_multipliers_method(func, constraint, x0):
    n = len(x0)

    def lagrange_system(vars):
        x = vars[:n]
        lam = vars[n]

        grad_f = approx_gradient(func, x)
        grad_phi = approx_gradient(constraint, x)

        equations = np.zeros(n + 1)
        equations[:n] = grad_f + lam * grad_phi
        equations[n] = constraint(x)
        return equations

    x0_full = np.append(x0, 0.0)
    sol, info, _, _ = fsolve(lagrange_system, x0_full, full_output=True)

    x_opt = sol[:n]
    lam_opt = sol[n]
    return x_opt, func(x_opt), lam_opt

In [56]:
res_x, res_f, res_lam = lagrange_multipliers_method(f, constraint, x0)
print(res_x, res_f)

[13.58877115  4.29438544] 9.294385703309057


## Метод Зойтендейка

In [57]:
def zoutendijk(f, constraint, x0, eps=1e-6, max_iter=100):
    x = np.array(x0, dtype=float)
    n = len(x)

    for k in range(max_iter):
        grad_f = approx_gradient(f, x)
        g_val = constraint(x)
        grad_g = approx_gradient(constraint, x)

        c = np.zeros(n + 1)
        c[-1] = 1.0

        A_ub = np.zeros((1, n + 1))
        A_ub[0, :n] = grad_f
        A_ub[0, -1] = -1.0
        b_ub = np.array([0.0])

        A_eq = np.zeros((1, n + 1))
        A_eq[0, :n] = grad_g
        A_eq[0, -1] = 0.0
        b_eq = np.array([0.0])

        bounds = [(-1.0, 1.0) for _ in range(n)] + [(None, None)]

        res = linprog(c, A_ub=A_ub, b_ub=b_ub,
                      A_eq=A_eq, b_eq=b_eq,
                      bounds=bounds, method='highs')

        d = res.x[:n]
        beta = res.x[-1]
        if beta >= -eps:
            break

        def phi(alpha):
            return f(x + alpha * d)

        res_line = minimize_scalar(phi, bounds=(0, 1e10), method='bounded')
        alpha_opt = res_line.x
        x_new = x + alpha_opt * d

        if np.linalg.norm(x_new - x) < eps * (1 + np.linalg.norm(x)):
            x = x_new
            break

        x = x_new

    return x, f(x)

In [58]:
res_x, res_f = zoutendijk(f, constraint, x0, max_iter=10)
print(res_x, res_f)

[-9.99999985e+09 -4.99999992e+09] -4999999924.39539


##  Метод Розена

In [59]:
def rosen(f, A, b, x0, eps=1e-6, max_iter=100):
    x = np.array(x0, dtype=float)
    n = len(x)

    for k in range(max_iter):
        AAT = A @ A.T
        inv_AAT = np.linalg.inv(AAT)
        P = np.eye(n) - A.T @ inv_AAT @ A

        grad_f = approx_gradient(f, x)
        S = -P @ grad_f

        norm_S = np.linalg.norm(S)
        if norm_S <= eps:
            lamb = inv_AAT @ A @ grad_f
            break

        alpha_max = 1e10

        def phi(alpha):
            return f(x + alpha * S)

        res = minimize_scalar(phi, bounds=(0, alpha_max), method='bounded')
        alpha_opt = res.x

        x_new = x + alpha_opt * S
        if np.linalg.norm(x_new - x) < eps * (1 + np.linalg.norm(x)):
            x = x_new
            break
        x = x_new

    return x, f(x)

In [60]:
res_x, res_f = rosen(f, A, b, x0)
print(res_x, res_f)

[-3.99999994e+09  1.19999998e+10] -15999999749.64355
